# Capstone: turn change events into Type 2 history

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/cdc-to-scd2-capstone.ipynb)

This small model follows a replayed change through a history build and point-in-time join.

## Setup and sample data

Two events apply to customer 42, and one is replayed. Event IDs are stable and each effective date has one change.

In [ ]:
changes = [(1, 42, "2026-01-01", "basic"), (2, 42, "2026-02-01", "plus"), (2, 42, "2026-02-01", "plus")]
orders = [(101, "2026-01-31"), (102, "2026-02-01")]

## Build half-open intervals

Deduplicate the replay by event ID, sort by effective time, and end each version where the next begins. End=None represents an open interval.

**Where it matters**

Give facts the dimension version that applied when they occurred.

**Watch out for**

Production feeds can contain conflicting events at the same effective time. Specify a source-sequence tie-breaker and an affected-history rebuild policy.

In [ ]:
unique_changes = {event[0]: event for event in changes}
versions = sorted(unique_changes.values(), key=lambda event: event[2])
history = [(start, versions[i + 1][2] if i + 1 < len(versions) else None, tier)
           for i, (_, _, start, tier) in enumerate(versions)]
print(history)

**What to notice:** Basic ends on February 1, when plus begins. The duplicated event produces no extra version.

## Resolve orders point in time

Use an inclusive start and exclusive end to assign orders to versions. The fixture is scoped to customer 42.

**Where it matters**

Attribute historical measures to their correct category.

**Watch out for**

A join on customer ID alone can match multiple versions. Check that every fact resolves to exactly one version.

In [ ]:
for order_id, ordered_at in orders:
    print(order_id, [tier for start, end, tier in history if start <= ordered_at and (end is None or ordered_at < end)])

**What to notice:** Order 101 matches basic and order 102 matches plus exactly on the boundary.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Which version would apply to an order on January 1?

In [ ]:
# Try your solution here.

In [ ]:
print("basic")

## Key takeaways

- Give facts the dimension version that applied when they occurred.
- Attribute historical measures to their correct category.

**Next:** [Data modeling 3: Slowly changing dimensions](../data-modeling/data-modeling-03-slowly-changing-dimensions.ipynb)